In [5]:
from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithoutReference
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from datasets import Dataset
import mlflow
from src.agent.graph import agent
from src.core.config import settings

/var/folders/n1/2qvnq4v53p51hj0p1nsg6ldc0000gn/T/ipykernel_67716/770387039.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithoutReference
/var/folders/n1/2qvnq4v53p51hj0p1nsg6ldc0000gn/T/ipykernel_67716/770387039.py:3: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithoutReference
/var/folders/n1/2qvnq4v53p51hj0p1nsg6ldc0000gn/T/ipykernel_67716/770387039.py:3: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in

In [7]:
test_queries = [
    "Does regular aerobic exercise reduce C-reactive protein levels in patients with type 2 diabetes?",
    "What is the efficacy of PD-1 inhibitors compared to traditional chemotherapy in metastatic melanoma?",
    "Are there significant correlations between gut microbiome diversity and the severity of major depressive disorder?",
    "What are the long-term cardiovascular risks associated with the use of second-generation antipsychotics in adolescents?",
    "How does CRISPR-Cas9 gene editing efficiency vary across different human hematopoietic stem cell lines?",
    "What is the impact of telemedicine interventions on glycemic control in rural hypertensive populations?",
    "Does maternal vitamin D supplementation during pregnancy reduce the incidence of childhood asthma?",
    "What are the molecular mechanisms by which resveratrol influences SIRT1 expression in aging murine models?",
    "Is there a higher risk of postoperative infection in robotic-assisted laparoscopic prostatectomy vs open surgery?",
    "What is the diagnostic sensitivity of liquid biopsies for detecting early-stage non-small cell lung cancer?"
]

# Build the dataset from pipeline results
data = {
    "question": [],
    "answer": [],
    "contexts": [],
}

# Run test queries through the agent
for query in test_queries:
    result = agent.invoke({
            "api_key": None,
            "query": query,
            "search_query": None,
            "abstracts": [],
            "llm_response": None,
            "claims": None,
            "scored_claims": None,
            "confidence_score": None,
            "final_response": None,
            "has_drug_query": False,
            "drug_names": None,
            "drug_labels": None,
            "has_fhir": None,
            "fhir_resource_type": None,
            "fhir_resource_id": None,
            "fhir_output": None
        })
    final = result["final_response"]
    data["question"].append(query)
    data["answer"].append(final["response"])
    data["contexts"].append([a["abstract"] for a in final["abstracts"]])

dataset = Dataset.from_dict(data)

/Users/andrewfranco/Documents/JetBrains/Pycharm/SentinelMD/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/20 23:44:30 INFO mlflow.tracking.fluent: Experiment with name 'SentinelMD' does not exist. Creating a new experiment.


FDA lookup failed for PD-1: 404 Client Error: Not Found for url: https://api.fda.gov/drug/label.json?search=openfda.brand_name:%22PD-1%22&limit=1
FDA lookup failed for CTLA-4: 404 Client Error: Not Found for url: https://api.fda.gov/drug/label.json?search=openfda.brand_name:%22CTLA-4%22&limit=1
FDA lookup failed for LAG-3: 404 Client Error: Not Found for url: https://api.fda.gov/drug/label.json?search=openfda.brand_name:%22LAG-3%22&limit=1
FDA lookup failed for EGFR-TKI: 404 Client Error: Not Found for url: https://api.fda.gov/drug/label.json?search=openfda.brand_name:%22EGFR-TKI%22&limit=1
FDA lookup failed for EGFR: 404 Client Error: Not Found for url: https://api.fda.gov/drug/label.json?search=openfda.brand_name:%22EGFR%22&limit=1
FDA lookup failed for ALK: 404 Client Error: Not Found for url: https://api.fda.gov/drug/label.json?search=openfda.brand_name:%22ALK%22&limit=1
FDA lookup failed for KRAS: 404 Client Error: Not Found for url: https://api.fda.gov/drug/label.json?search=open

In [8]:
custom_run_config = RunConfig(
    max_workers=1,
    max_retries=15,
    timeout=120 # Gives the API more time to respond before failing
)

llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(model="gemma-3-27b-it", google_api_key=settings.GEMINI_API_KEY))
embeddings = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", google_api_key=settings.GEMINI_API_KEY))

results = evaluate(
    dataset=dataset,
    metrics=[
        Faithfulness(),
        ResponseRelevancy(),
        LLMContextPrecisionWithoutReference()
    ],
    llm=llm,
    embeddings=embeddings,
    run_config=custom_run_config
)
print(results)

/var/folders/n1/2qvnq4v53p51hj0p1nsg6ldc0000gn/T/ipykernel_67716/2330226121.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(model="gemma-3-27b-it", google_api_key=settings.GEMINI_API_KEY))
/var/folders/n1/2qvnq4v53p51hj0p1nsg6ldc0000gn/T/ipykernel_67716/2330226121.py:8: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", google_api_key=settings.GEMINI_API_KEY))
Evaluating: 10

{'faithfulness': 0.7735, 'answer_relevancy': nan, 'llm_context_precision_without_reference': 0.5609}


In [ ]:
with mlflow.start_run(run_name="ragas_evaluation"):
    mlflow.log_metric("faithfulness", 0.9863)
    mlflow.log_metric("context_precision", 1.0)
    mlflow.log_param("answer_relevancy", "timeout_error")
    mlflow.log_param("num_queries", len(test_queries))
    mlflow.log_param("model", "gemma-4-31b-it")
    mlflow.log_param("retrieval_k", 5)